# Stage 1 — Preprocessing & Contrast Enhancement

Pipeline: **green channel extraction → CLAHE → morphological top-hat**.

This notebook imports the reusable functions from `src/preprocessing.py` (kept in a plain module so Stage 2 can `import` them too) and is used purely to run them interactively, visualize every intermediate step, and tune parameters.

**Before running**: download DRIVE (and optionally STARE / CHASE_DB1) and place images under `data/`, e.g.:
```
data/drive/images/21_training.tif
data/drive/mask/21_training_mask.gif
```
Update `IMAGE_PATH` / `MASK_PATH` below to point at a real file before running the load cell.

In [ ]:
import sys
from pathlib import Path

# Allow `from src...` imports when running the notebook from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.io_utils import load_fundus_image, load_fov_mask
from src.preprocessing import extract_green_channel, apply_clahe, tophat_filter, run_stage1

In [ ]:
# --- Point these at a real DRIVE image + mask before running ---
IMAGE_PATH = str(PROJECT_ROOT / "data" / "drive" / "images" / "21_training.tif")
MASK_PATH = str(PROJECT_ROOT / "data" / "drive" / "mask" / "21_training_mask.gif")
# STARE / CHASE_DB1 don't ship a mask -- pass MASK_PATH = None for those.

rgb = load_fundus_image(IMAGE_PATH)
fov_mask = load_fov_mask(MASK_PATH)
print("Image shape:", rgb.shape, "| Mask:", None if fov_mask is None else fov_mask.shape)

In [ ]:
# --- Tunable stage-1 parameters ---
CLIP_LIMIT = 2.0
TILE_GRID_SIZE = (8, 8)   # scale up (e.g. (10,10)-(12,12)) for STARE/CHASE_DB1's larger frames
DISK_RADIUS = 9           # must exceed the widest vessel (~1-15 px in these datasets)

green = extract_green_channel(rgb)
enhanced = apply_clahe(green, clip_limit=CLIP_LIMIT, tile_grid_size=TILE_GRID_SIZE)
tophat = tophat_filter(enhanced, disk_radius=DISK_RADIUS)
masked = np.where(fov_mask > 0, tophat, 0).astype(np.uint8) if fov_mask is not None else tophat

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
panels = [
    (rgb, "Original RGB", None),
    (green, "Green channel", "gray"),
    (enhanced, "Post-CLAHE", "gray"),
    (masked, "Post-top-hat (masked)", "gray"),
]
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Visual checks before moving to Stage 2
- **Post-CLAHE**: illumination should look flat across the retina (no bright center / dark edges); no visible noise "speckling" in flat background (if speckling appears, lower `CLIP_LIMIT`).
- **Post-top-hat**: optic disc should be almost fully suppressed; thin vessels near the FOV edge should still be visible (if the optic disc still shows a bright blob, increase `DISK_RADIUS`; if thin vessels vanish, decrease it).
- **With the FOV mask applied**: the black surround must be uniformly zero, no bright ring artifact at the boundary.

In [ ]:
# Optional: live parameter tuning with sliders (requires ipywidgets)
from ipywidgets import interact, FloatSlider, IntSlider

@interact(
    clip_limit=FloatSlider(value=CLIP_LIMIT, min=0.5, max=6.0, step=0.5),
    disk_radius=IntSlider(value=DISK_RADIUS, min=3, max=25, step=1),
)
def tune(clip_limit=CLIP_LIMIT, disk_radius=DISK_RADIUS):
    result = run_stage1(
        rgb, fov_mask,
        clip_limit=clip_limit,
        tile_grid_size=TILE_GRID_SIZE,
        disk_radius=disk_radius,
    )
    plt.figure(figsize=(6, 6))
    plt.imshow(result, cmap="gray")
    plt.axis("off")
    plt.title(f"clip_limit={clip_limit}, disk_radius={disk_radius}")
    plt.show()

In [ ]:
# Save the 4-panel comparison figure for later reference / reports
from pathlib import Path

out_dir = PROJECT_ROOT / "outputs" / "stage1"
out_dir.mkdir(parents=True, exist_ok=True)

image_id = Path(IMAGE_PATH).stem
fig.savefig(out_dir / f"{image_id}_stages.png", dpi=150, bbox_inches="tight")
print("Saved to", out_dir / f"{image_id}_stages.png")